In [ ]:
import os
import pandas as pd

from _plot_utils import plot_lift
from pass_pclr.defines import PTBXL_TARGETS


def load_ptbxl_experiment_data(runs_dir):
    experiments = {}

    # Get all experiment directories
    for exp_dir in sorted(os.listdir(runs_dir)):
        exp_path = os.path.join(runs_dir, exp_dir)

        # Check if it's a directory
        if os.path.isdir(exp_path):
            metrics_file = os.path.join(exp_path, "metrics.csv")
            probs_file = os.path.join(exp_path, "probs.npy")

            # Check if metrics.csv exists
            if os.path.exists(metrics_file):
                df = pd.read_csv(metrics_file)

                # Get AUROC values for multilabel tasks
                multilabel_aurocs = []
                multilabel_auprcs = []
                for task in PTBXL_TARGETS:
                    task_data = df[df["Label"] == task].iloc[0]
                    multilabel_aurocs.append(task_data["AUROC"])
                    multilabel_auprcs.append(task_data["AUPRC"])

                multilabel_avg_data = df[df["Label"] == "Multilabel Averaged"].iloc[0]

                experiments[exp_dir] = {
                    "multilabel_aurocs": multilabel_aurocs,
                    "multilabel_avg_auroc": multilabel_avg_data["AUROC"],
                    "multilabel_avg_auprc": multilabel_avg_data["AUPRC"],
                    "all_data": df,
                }

    return experiments

In [ ]:
experiments = {
    "full": load_ptbxl_experiment_data("../outputs/runs-ptbxl/"),
    "8k": load_ptbxl_experiment_data("../outputs/runs-ptbxl-8k/"),
    "4k": load_ptbxl_experiment_data("../outputs/runs-ptbxl-4k/"),
    "2k": load_ptbxl_experiment_data("../outputs/runs-ptbxl-2k/"),
}

print("Experiments found:")
for exp_subset, subset_results in experiments.items():
    print(f"\tSubset {exp_subset}:")
    for exp_name, exp_data in subset_results.items():
        print(f"\t\t{exp_name}: multilabel AUROC = {exp_data['multilabel_avg_auroc']:.4f}")

In [ ]:
sizes = {
    "full": 17418,
    "8k": 8722,
    "4k": 4356,
    "2k": 2175,
}

aliases = {
    "proto-from-scratch-v2": "proto-from-scratch",
    "pass-heedb-pip": "pass-heedb-pip",
    "pass-heedb-pit": "pass-heedb-pit",
    "pass-heedb-pip-logreg": "pass-heedb-pip-logreg",
    "pass-heedb-pit-logreg": "pass-heedb-pit-logreg",
}

palette = {
    "proto-from-scratch": "tab:brown",
    "pass-heedb-pip": "tab:blue",
    "pass-heedb-pit": "tab:orange",
    "pass-heedb-pip-logreg": "tab:purple",
    "pass-heedb-pit-logreg": "tab:gray",
}

df = pd.DataFrame.from_records(
    [
        {
            "Model": aliases[exp_name],
            "Train Size": sizes[exp_subset],
            "Multilabel (AUROC)": exp_data["multilabel_avg_auroc"],
            "Multilabel (AUPRC)": exp_data["multilabel_avg_auprc"],
        }
        for exp_subset, subset_results in experiments.items()
        for exp_name, exp_data in subset_results.items()
        if exp_name in aliases
    ]
)

In [ ]:
plot_lift(
    data=df,
    metric="Multilabel (AUROC)",
    palette=palette,
    title="Multilabel Averaged (AUROC)",
    save_path="figs/ptbxl-multilabel-lift-roc.png",
    ylim=(0.45, 0.9),
)
plot_lift(
    data=df,
    metric="Multilabel (AUPRC)",
    palette=palette,
    title="Multilabel Averaged (AUPRC)",
    save_path="figs/ptbxl-multilabel-lift-pr.png",
    ylim=(0.05, 0.4),
)